## **PubMed SQLite validation**

Este notebook importa e valida as funções de `src/bibliometric_rag_assistant/db.py`, originadas da etapa exploratória no notebook `02_pubmed_sqlite_persistence_exploration.ipynb`.

A execução consulta o PubMed e grava uma nova execução no banco local.

In [1]:
import json
import sqlite3
from datetime import datetime, timezone

from bibliometric_rag_assistant.config import DB_PATH, RAW_PUBMED_DIR
from bibliometric_rag_assistant.db import (
    connect_database,
    count_rows,
    create_tables,
    save_ingestion,
)
from bibliometric_rag_assistant.ingestion.pubmed import (
    fetch_pubmed_citations,
    parse_pubmed_xml,
    search_pubmed,
)
from bibliometric_rag_assistant.search_strategy import (
    PUBMED_ACTIVE_VARIANT,
    PUBMED_QUERY,
    PUBMED_SAMPLE_SIZE,
    PUBMED_SOURCE,
    PUBMED_VARIANTS,
)

print("Database:", DB_PATH)

Database: /home/lucas/github/bibliometric-rag-assistant/data/interim/bibliometria.db


### **Conexão e criação das tabelas**

In [2]:
connection = connect_database()
create_tables(connection)

counts_before = count_rows(connection)
counts_before

{'ingestion_runs': 0, 'raw_snapshots': 0, 'citations': 0}

### **Busca no PubMed e montagem da execução**

In [3]:
now = datetime.now(timezone.utc)

search_result = search_pubmed(PUBMED_QUERY, PUBMED_SAMPLE_SIZE)
esearch = search_result["esearchresult"]
pmids = esearch["idlist"]

run = {
    "run_id": now.strftime("%Y%m%dT%H%M%SZ"),
    "source": PUBMED_SOURCE,
    "query_text": PUBMED_QUERY,
    "query_variant": PUBMED_ACTIVE_VARIANT,
    "query_version": PUBMED_VARIANTS[PUBMED_ACTIVE_VARIANT]["version"],
    "retrieved_at": now.isoformat(),
    "total_results": int(esearch["count"]),
    "total_retrieved": len(pmids),
}

run

{'run_id': '20261008T225808Z',
 'source': 'pubmed',
 'query_text': '( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/01"[Date - Publication] : "2025/12/31"[Date - Publication] )',
 'query_variant': 'query_provisoria_atual',
 'query_version': '0.1',
 'retrieved_at': '2026-10-08T22:58:08.445137+00:00',
 'total_results': 100,
 'total_retrieved': 20}

### **Coleta das citações e snapshots brutos**

In [4]:
search_path = RAW_PUBMED_DIR / f"{run['run_id']}_esearch.json"
search_path.write_text(
    json.dumps(search_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

pubmed_xml = fetch_pubmed_citations(pmids)

fetch_path = RAW_PUBMED_DIR / f"{run['run_id']}_efetch.xml"
fetch_path.write_text(pubmed_xml, encoding="utf-8")

citations = parse_pubmed_xml(pubmed_xml)

snapshots = [
    {
        "run_id": run["run_id"],
        "endpoint": "esearch",
        "payload_format": "json",
        "payload": search_result,
        "file_path": search_path,
    },
    {
        "run_id": run["run_id"],
        "endpoint": "efetch",
        "payload_format": "xml",
        "payload": pubmed_xml,
        "file_path": fetch_path,
    },
]

len(citations)

20

### **Gravação no banco**

In [5]:
save_ingestion(connection, run, snapshots, citations)

counts_after = count_rows(connection)
counts_after

{'ingestion_runs': 1, 'raw_snapshots': 2, 'citations': 20}

### **Validação**

In [6]:
assert counts_after["ingestion_runs"] == counts_before["ingestion_runs"] + 1
assert counts_after["raw_snapshots"] == counts_before["raw_snapshots"] + len(snapshots)
assert counts_after["citations"] == counts_before["citations"] + len(citations)

stored_run = connection.execute(
    """
    SELECT run_id, source, query_variant, query_version, total_results, total_retrieved
    FROM ingestion_runs
    WHERE run_id = ?
    """,
    (run["run_id"],),
).fetchone()

stored_run

('20261008T225808Z', 'pubmed', 'query_provisoria_atual', '0.1', 100, 20)

In [7]:
run["run_id"]

'20261008T225808Z'

In [8]:
connection.execute(
    """
    SELECT endpoint, payload_format, LENGTH(payload), file_path
    FROM raw_snapshots
    WHERE run_id = ?
    """,
    (run["run_id"],),
).fetchall()

[('efetch',
  'xml',
  253969,
  '/home/lucas/github/bibliometric-rag-assistant/data/raw/pubmed/20261008T225808Z_efetch.xml'),
 ('esearch',
  'json',
  797,
  '/home/lucas/github/bibliometric-rag-assistant/data/raw/pubmed/20261008T225808Z_esearch.json')]

Os autores e as afiliações são gravados como JSON e devem voltar iguais às listas originais.

In [9]:
pmid, title, publication_year, authors_json, affiliations_json = connection.execute(
    """
    SELECT pmid, title, publication_year, authors_json, affiliations_json
    FROM citations
    WHERE run_id = ?
    ORDER BY pmid
    LIMIT 1
    """,
    (run["run_id"],),
).fetchone()

original = next(citation for citation in citations if citation["pmid"] == pmid)

assert json.loads(authors_json) == original["authors"]
assert json.loads(affiliations_json) == original["affiliations"]

pmid, publication_year, title

('37246472',
 2024,
 '3D-printed temporomandibular joint-mandible combined prosthesis: A prospective study.')

Gravar a mesma execução de novo deve falhar pela chave primária e não deixar nada pela metade.

In [10]:
try:
    save_ingestion(connection, run, snapshots, citations)
except sqlite3.IntegrityError as error:
    print("Rejeitado como esperado:", error)
else:
    raise AssertionError("A execução duplicada deveria ter sido rejeitada")

assert count_rows(connection) == counts_after

Rejeitado como esperado: UNIQUE constraint failed: ingestion_runs.run_id


Não pode haver snapshot ou citação apontando para uma execução inexistente.

In [11]:
assert connection.execute("PRAGMA foreign_key_check").fetchall() == []

In [12]:
connection.close()